In [ ]:
# %% [markdown]
# # Method 2: pair approximation for the spatial reputation Q-learning model
#
# This notebook-style script follows the pair-approximation idea used for
# interaction-state Q-learning on a square lattice.
#
# Instead of tracking only rho_C, we track local spatial correlations:
#
# - p_CC: probability that a directed edge points from C to C
# - p_CD: probability that a directed edge points from C to D
# - p_DC = p_CD by symmetry
# - p_DD: probability that a directed edge points from D to D
#
# Constraint:
#
# p_CC + 2 p_CD + p_DD = 1
#
# Node density:
#
# rho_C = p_CC + p_CD

# %%
import math
import numpy as np
import matplotlib.pyplot as plt

# 设置图形样式 - 背景纯白
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'


# %% [markdown]
# ## 1. Local reward and epsilon-greedy choice

# %%
def cooperation_advantage(x, rbar):
    """Expected shaped-fitness advantage Delta F = F_C - F_D."""
    return -(1.0 - rbar) * x + rbar * (1.0 + x) / 4.0


def choose_prob_c(delta_f, epsilon=0.02, beta=20.0, decision="soft"):
    """Probability of choosing C under an epsilon-greedy approximation."""
    if decision == "hard":
        return 1.0 - epsilon / 2.0 if delta_f > 0.0 else epsilon / 2.0
    if decision == "soft":
        return epsilon / 2.0 + (1.0 - epsilon) / (1.0 + math.exp(-beta * delta_f))
    raise ValueError("decision must be 'soft' or 'hard'")


def binom_pmf(k, n, q):
    q = min(max(q, 0.0), 1.0)
    return math.comb(n, k) * (q**k) * ((1.0 - q) ** (n - k))


# %% [markdown]
# ## 2. Pair-approximation ODE
#
# We use undirected edge densities internally:
#
# - E_CC
# - E_CD
# - E_DD
#
# with E_CC + E_CD + E_DD = 1.
#
# The relation to the directed notation above is:
#
# p_CC = E_CC
# p_CD = E_CD / 2
# p_DD = E_DD
#
# Thus rho_C = E_CC + E_CD/2.
#
# If a C node with k cooperative neighbors switches to D:
#
# - k CC edges become CD
# - 4-k CD edges become DD
#
# If a D node with k cooperative neighbors switches to C:
#
# - k CD edges become CC
# - 4-k DD edges become CD

# %%
def pair_derivative(
    y,
    x=0.3,
    c=0.1,
    c_max=1.0,
    epsilon=0.02,
    beta=20.0,
    decision="soft",
    reputation_mode="stationary",
    R_C=None,
    R_D=None,
):
    """Derivative for undirected pair densities [E_CC, E_CD, E_DD].

    reputation_mode='stationary':
        assume persistent cooperators have R_C=c_max and defectors have R_D=0.

    reputation_mode='given':
        use user-supplied R_C and R_D.
    """
    E_CC, E_CD, E_DD = y
    E_CC = max(E_CC, 1e-12)
    E_CD = max(E_CD, 1e-12)
    E_DD = max(E_DD, 1e-12)
    norm = E_CC + E_CD + E_DD
    E_CC, E_CD, E_DD = E_CC / norm, E_CD / norm, E_DD / norm

    rho_C = E_CC + 0.5 * E_CD
    rho_D = 1.0 - rho_C

    # Conditional probabilities for a neighbor being C.
    q_C = 2.0 * E_CC / max(2.0 * E_CC + E_CD, 1e-12)
    q_D = E_CD / max(E_CD + 2.0 * E_DD, 1e-12)

    if reputation_mode == "stationary":
        R_C_eff = c_max
        R_D_eff = 0.0
    elif reputation_mode == "given":
        if R_C is None or R_D is None:
            raise ValueError("R_C and R_D are required when reputation_mode='given'")
        R_C_eff = R_C
        R_D_eff = R_D
    else:
        raise ValueError("reputation_mode must be 'stationary' or 'given'")

    dE_CC = 0.0
    dE_CD = 0.0
    dE_DD = 0.0

    # C -> D events.
    for k in range(5):
        prob_k = binom_pmf(k, 4, q_C)
        rbar = (k * R_C_eff + (4 - k) * R_D_eff) / (4.0 * c_max)
        delta_f = cooperation_advantage(x, rbar)
        prob_choose_d = 1.0 - choose_prob_c(delta_f, epsilon, beta, decision)
        rate = rho_C * prob_k * prob_choose_d

        dE_CC += rate * (-k / 4.0)
        dE_CD += rate * ((2.0 * k - 4.0) / 4.0)
        dE_DD += rate * ((4.0 - k) / 4.0)

    # D -> C events.
    for k in range(5):
        prob_k = binom_pmf(k, 4, q_D)
        rbar = (k * R_C_eff + (4 - k) * R_D_eff) / (4.0 * c_max)
        delta_f = cooperation_advantage(x, rbar)
        prob_choose_c = choose_prob_c(delta_f, epsilon, beta, decision)
        rate = rho_D * prob_k * prob_choose_c

        dE_CC += rate * (k / 4.0)
        dE_CD += rate * ((4.0 - 2.0 * k) / 4.0)
        dE_DD += rate * (-(4.0 - k) / 4.0)

    return np.array([dE_CC, dE_CD, dE_DD])


def normalize_edges(y):
    y = np.maximum(np.asarray(y, dtype=float), 1e-12)
    return y / y.sum()


def rho_from_edges(y):
    E_CC, E_CD, _ = normalize_edges(y)
    return E_CC + 0.5 * E_CD


# %% [markdown]
# ## 3. Integrator

# %%
def integrate_pair_approximation(
    x=0.3,
    c=0.1,
    c_max=1.0,
    epsilon=0.02,
    beta=20.0,
    decision="soft",
    reputation_mode="stationary",
    steps=5000,
    dt=0.02,
    initial_rho=0.1,
    initial_mixing="random",
    R_C0=1,
    R_D0=0,
):
    if initial_mixing == "random":
        E_CC = initial_rho**2
        E_CD = 2.0 * initial_rho * (1.0 - initial_rho)
        E_DD = (1.0 - initial_rho) ** 2
    elif initial_mixing == "checkerboard":
        E_CC = 0.02
        E_CD = 0.96
        E_DD = 0.02
    else:
        raise ValueError("initial_mixing must be 'random' or 'checkerboard'")

    y = normalize_edges([E_CC, E_CD, E_DD])
    R_C = c_max if R_C0 is None else float(R_C0)
    R_D = 0.0 if R_D0 is None else float(R_D0)
    series = np.empty((steps, 6))

    for t in range(steps):
        derivative_reputation_mode = "stationary"
        if reputation_mode == "dynamic":
            derivative_reputation_mode = "given"
        elif reputation_mode != "stationary":
            raise ValueError("reputation_mode must be 'stationary' or 'dynamic'")

        dy = pair_derivative(
            y,
            x=x,
            c=c,
            c_max=c_max,
            epsilon=epsilon,
            beta=beta,
            decision=decision,
            reputation_mode=derivative_reputation_mode,
            R_C=R_C,
            R_D=R_D,
        )
        y = normalize_edges(y + dt * dy)

        if reputation_mode == "dynamic":
            # Semi-analytical first-order reputation closure:
            # persistent C tends to c_max, persistent D tends to 0.
            # This captures the speed effect of c without tracking a full
            # reputation distribution P(strategy, reputation).
            R_C = np.clip(R_C + dt * c * (1.0 - R_C / c_max), 0.0, c_max)
            R_D = np.clip(R_D - dt * c * (R_D / c_max), 0.0, c_max)

        series[t, :3] = y
        series[t, 3] = rho_from_edges(y)
        series[t, 4] = R_C
        series[t, 5] = R_D

    return series


# %% [markdown]
# ## 4. Sweep x (dilemma strength) vs ρ_C (cooperation fraction)

# %%
def sweep_pair_x(
    x_values,
    c=0.1,
    c_max=1.0,
    epsilon=0.02,
    beta=25.0,
    decision="soft",
    reputation_mode="dynamic",
    steps=10000,
    burn=9500,
):
    """Sweep over x values and return steady-state cooperation fraction."""
    rho_star = np.empty_like(x_values, dtype=float)
    ecd_star = np.empty_like(x_values, dtype=float)
    
    for idx, x in enumerate(x_values):
        # print(f"Computing x = {x:.3f}...")
        s = integrate_pair_approximation(
            x=x,
            c=c,
            c_max=c_max,
            epsilon=epsilon,
            beta=beta,
            decision=decision,
            reputation_mode=reputation_mode,
            steps=steps,
        )
        rho_star[idx] = s[burn:, 3].mean()
        ecd_star[idx] = s[burn:, 1].mean()
    
    return rho_star, ecd_star


# 参数设置
epsilon = 0.01  # 探索率
c = 0.8  # 声誉更新步长
beta = 25.0  # 决策温度

# 扫描x从0到1
x_values = np.linspace(0.01, 1.0, 40)
rho_star, ecd_star = sweep_pair_x(x_values, c=c, epsilon=epsilon, beta=beta)

# 图1: x vs ρ_C
fig1, (ax1) = plt.subplots(1, 1, figsize=(6, 5))
fig1.patch.set_facecolor('white')
ax1.set_facecolor('white')

# 设置坐标轴边框（显示所有四条边）
ax1.spines['top'].set_visible(True)
ax1.spines['right'].set_visible(True)
ax1.spines['bottom'].set_visible(True)
ax1.spines['left'].set_visible(True)
ax1.spines['top'].set_color('black')
ax1.spines['right'].set_color('black')
ax1.spines['bottom'].set_color('black')
ax1.spines['left'].set_color('black')
ax1.spines['top'].set_linewidth(1.0)
ax1.spines['right'].set_linewidth(1.0)
ax1.spines['bottom'].set_linewidth(1.0)
ax1.spines['left'].set_linewidth(1.0)

ax1.plot(x_values, rho_star, 'b-', linewidth=2, label=f'c={c}', marker='o', markerfacecolor='none', markeredgecolor='blue')
ax1.set_xlabel("$b$", fontsize=24)
ax1.set_ylabel(r" $f_C$", fontsize=24)
ax1.set_ylim(-0.05, 1.05)
ax1.set_xlim(0, 1)
ax1.grid(False)
ax1.legend(loc='best', fontsize=18, frameon=False)

# 设置刻度显示
ax1.tick_params(axis='both', which='major', labelsize=14, direction='in', length=5, width=1.0)
ax1.tick_params(axis='both', which='minor', direction='in', length=3, width=0.8)

plt.tight_layout()
plt.show()

# fig1.savefig('Figure\\fc_b_pair.pdf', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
# fig1.savefig('Figure\\x_vs_rho_pair.png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
# print("图1已保存: Figure\\x_vs_rho_pair.png 和 Figure\\fc_b_pair.pdf")